# Timestamps y Fechas en Pandas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/14_pandas_transformaciones/code/02_timestamps.ipynb)

Las fechas son una de las fuentes de datos más comunes y más problemáticas. Vienen en mil formatos distintos, con o sin zona horaria, con ambigüedades que pueden arruinar un análisis. En este notebook aprendemos a manejarlas correctamente con pandas: parseo robusto, extracción de componentes, aritmética de fechas, y zonas horarias.

**Temas:**
- `pd.to_datetime()` y sus trampas
- `pd.Timestamp` y `DatetimeIndex`
- El accessor `.dt` para extraer año, mes, día, etc.
- UTC y timezones (por qué importa)
- Aritmética con `Timedelta` y `DateOffset`
- Features temporales para ML

In [ ]:
%pip install pandas==2.2.3 numpy==1.26.4 pytz==2024.1

In [ ]:
import pandas as pd
import numpy as np
import pytz
from io import StringIO

print(f"pandas: {pd.__version__}")
print(f"numpy:  {np.__version__}")
print(f"pytz:   {pytz.__version__}")

---
## Sección 1: El problema con fechas en datos reales

En el mundo real las fechas llegan de múltiples sistemas con formatos distintos. Una fecha puede estar en ISO 8601, en formato europeo, americano, como texto, como Unix timestamp... o simplemente ser inválida. El primer paso es siempre **parsear correctamente**.

### `pd.to_datetime()` — el parser universal

El parámetro clave es `errors`:
- `errors="raise"` (default): lanza excepción si hay un valor inválido
- `errors="coerce"`: los inválidos se convierten en `NaT` (Not a Time)
- `errors="ignore"`: devuelve el valor original sin convertir

In [ ]:
# Fechas tal como llegan de distintas fuentes
fechas_sucias = pd.Series([
    "2024-01-15",           # ISO 8601
    "15/01/2024",           # día/mes/año (europeo/latam)
    "01/15/2024",           # mes/día/año (americano)
    "Jan 15, 2024",         # texto
    "1705276800",           # Unix timestamp (segundos desde epoch)
    "2024-01-15 10:30:00",  # con hora
    "not_a_date",           # inválido
    None,                   # faltante
])

print("Serie original:")
print(fechas_sucias)
print(f"\nTipo: {fechas_sucias.dtype}")

In [ ]:
# errors="coerce" convierte lo que puede, pone NaT donde no puede
fechas_parseadas = pd.to_datetime(fechas_sucias, errors="coerce")

print("Después de to_datetime(errors='coerce'):")
print(fechas_parseadas)
print(f"\nTipo: {fechas_parseadas.dtype}")
print(f"NaTs: {fechas_parseadas.isna().sum()} de {len(fechas_parseadas)}")

Noten que el Unix timestamp `"1705276800"` se parseó como año 1705276800 — claramente incorrecto. Cuando sabes que una columna contiene Unix timestamps hay que especificarlo:

In [ ]:
# Parsear un Unix timestamp correctamente
unix_ts = pd.Series(["1705276800"])

# Convertir a entero primero, luego especificar la unidad
resultado = pd.to_datetime(unix_ts.astype(int), unit="s")
print(f"Unix 1705276800 → {resultado[0]}")

### La trampa de la ambigüedad de formato

`"01/02/2024"` — ¿es el 1 de febrero o el 2 de enero? Depende del sistema que generó el dato. Si no especificas el formato, pandas hace una suposición que puede estar mal.

**Regla:** cuando el formato no es ISO 8601, siempre usa `format=` explícito.

In [ ]:
fecha_ambigua = pd.Series(["01/02/2024", "03/04/2024", "05/06/2024"])

# pandas asume mes/día/año por default (convención americana)
resultado_auto = pd.to_datetime(fecha_ambigua, errors="coerce")

# Con formato europeo/latam explícito (día/mes/año)
resultado_latam = pd.to_datetime(fecha_ambigua, format="%d/%m/%Y")

# Con formato americano explícito (mes/día/año)
resultado_us = pd.to_datetime(fecha_ambigua, format="%m/%d/%Y")

comparacion = pd.DataFrame({
    "original": fecha_ambigua,
    "auto (asume EE.UU.)": resultado_auto,
    "format=%d/%m/%Y (latam)": resultado_latam,
    "format=%m/%d/%Y (EE.UU.)": resultado_us,
})
print(comparacion.to_string())
print("\n'01/02/2024' con formato latam es:", resultado_latam[0].strftime("%B %d, %Y"))
print("'01/02/2024' con formato EE.UU. es:", resultado_us[0].strftime("%B %d, %Y"))

> **💡 Prueba esto:** Cambia el string `"13/02/2024"` (13 de febrero — imposible como mes) y observa que pandas lanza un error con formato americano pero lo parsea correctamente con formato latam. ¿Qué pasa con `errors="coerce"`?

In [ ]:
x = pd.Series(["13/02/2024"])

pd.to_datetime(x,format="%m/,%d/,%Y/", errors="coerce")

#errors = "coerce", nos ayuda a parsear la fecha si es que nos confundimos entre utilizar el formato americano y el el otro
#en caso de que no lo pueda parsear, únicamente nos arrojara el NaT (Not a Time)

---
## Sección 2: `pd.Timestamp` y `DatetimeIndex`

`pd.Timestamp` es el tipo de dato de pandas para un punto en el tiempo (equivale a `datetime.datetime` de Python pero con más funcionalidad). Cuando tienes una Serie de Timestamps, se almacena como `DatetimeIndex` si es el índice, o como columna de tipo `datetime64[ns]`.

In [ ]:
# Crear Timestamps directamente
ahora = pd.Timestamp.now()
fecha_especifica = pd.Timestamp("2024-01-15")
fecha_con_hora = pd.Timestamp("2024-01-15 10:30:45")
fecha_componentes = pd.Timestamp(year=2024, month=6, day=21, hour=12)

print(f"Ahora:              {ahora}")
print(f"Fecha específica:   {fecha_especifica}")
print(f"Con hora:           {fecha_con_hora}")
print(f"Por componentes:    {fecha_componentes}")
print(f"Tipo:               {type(ahora)}")

In [ ]:
# pd.date_range — generar secuencias de fechas
# freq="MS" = Month Start (primer día de cada mes)
meses_2024 = pd.date_range("2024-01-01", periods=12, freq="MS")
print("12 meses del 2024 (MS = inicio de mes):")
print(meses_2024)
print(f"\nTipo: {type(meses_2024)}")

In [ ]:
# Diferentes frecuencias
frecuencias = {
    "D  (diario)": pd.date_range("2024-01-01", periods=5, freq="D"),
    "W  (semanal)": pd.date_range("2024-01-01", periods=5, freq="W"),
    "MS (inicio mes)": pd.date_range("2024-01-01", periods=5, freq="MS"),
    "QS (inicio trimestre)": pd.date_range("2024-01-01", periods=5, freq="QS"),
    "YS (inicio año)": pd.date_range("2024-01-01", periods=5, freq="YS"),
}

for nombre, rango in frecuencias.items():
    print(f"\n{nombre}:")
    print(" ", list(rango.strftime("%Y-%m-%d")))

> **💡 Prueba esto:** Genera un `date_range` con `freq="BMS"` (Business Month Start — inicio de mes en día hábil) y compara con `freq="MS"`. ¿En qué meses difieren en 2024?

In [ ]:
ms  = pd.date_range("2024-01-01", "2024-12-01", freq="MS")
bms = pd.date_range("2024-01-01", "2024-12-01", freq="BMS")

pd.DataFrame({"MS": ms, "BMS": bms}).query("MS != BMS")

---
## Sección 3: El accessor `.dt` — extraer componentes

Cuando tienes una columna de tipo `datetime64`, el accessor `.dt` te da acceso a todos los componentes de la fecha. Es la forma vectorizada (rápida) de hacer lo que harías con un loop.

In [ ]:
# DataFrame de ejemplo: ventas con timestamp
np.random.seed(42)
n = 20

fechas_str = pd.date_range(
    "2024-01-01 08:00", periods=n, freq="11D 3h 15min"
).strftime("%Y-%m-%d %H:%M:%S")

df = pd.DataFrame({
    "fecha_str": fechas_str,
    "ventas": np.random.randint(100, 5000, size=n),
    "sucursal": np.random.choice(["CDMX", "MTY", "GDL"], size=n),
})

print("DataFrame original:")
print(df.head())
print(f"\nTipo de fecha_str: {df['fecha_str'].dtype}")

In [ ]:
# Convertir a datetime y extraer todos los componentes útiles
df["fecha"] = pd.to_datetime(df["fecha_str"])

df["año"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month
df["dia"] = df["fecha"].dt.day
df["dia_semana"] = df["fecha"].dt.dayofweek        # 0=lunes, 6=domingo
df["nombre_dia"] = df["fecha"].dt.day_name()        # "Monday", "Tuesday"...
df["nombre_mes"] = df["fecha"].dt.month_name()      # "January", "February"...
df["trimestre"] = df["fecha"].dt.quarter            # 1, 2, 3 o 4
df["semana_año"] = df["fecha"].dt.isocalendar().week.astype(int)  # semana ISO
df["hora"] = df["fecha"].dt.hour

cols_mostrar = ["fecha", "año", "mes", "dia", "dia_semana",
                "nombre_dia", "nombre_mes", "trimestre", "semana_año", "hora"]
print(df[cols_mostrar].to_string())

In [ ]:
# Variable booleana: ¿es fin de semana?
# dayofweek: 0=lun, 1=mar, 2=mie, 3=jue, 4=vie, 5=sab, 6=dom
df["es_fin_semana"] = df["fecha"].dt.dayofweek >= 5

print("Distribución fin de semana vs semana:")
print(df["es_fin_semana"].value_counts())
print()
print("Ventas promedio por tipo de día:")
print(df.groupby("es_fin_semana")["ventas"].mean().rename({False: "Semana", True: "Fin de semana"}))

### Por qué esto importa para ML

Los modelos de ML no pueden trabajar directamente con timestamps. Necesitas convertirlos a features numéricas. Las que extrajimos arriba son el punto de partida: `día_semana`, `mes`, `hora`, `trimestre` capturan patrones de estacionalidad y ciclicidad.

**Ojo:** estas variables son cíclicas. El mes 12 (diciembre) NO está "lejos" del mes 1 (enero) — están a un paso. Los modelos de árbol (Random Forest, XGBoost) manejan esto bien, pero los modelos lineales no. Para modelos lineales, conviene usar encodings seno/coseno. Lo veremos más adelante.

> **💡 Prueba esto:** ¿Cómo crearías una variable `"es_inicio_mes"` que sea `True` cuando el día es 1? ¿Y `"es_quincena"` que sea `True` cuando el día es 1 o 15?

In [ ]:
df["es_inicioMes"] = df["fecha"].dt.day.eq(1)
df["es_quincena"] = df["fecha"].dt.day.isin([1,15])

---
## Sección 4: UTC y Timezones — por qué importa

Un timestamp sin zona horaria es ambiguo. Las 10:00 AM, ¿de dónde? Si tienes eventos de CDMX y de Nueva York sin normalizar, estás comparando peras con manzanas.

**Regla de oro:** guarda todo internamente en UTC, convierte a hora local solo para mostrar al usuario.

Dos operaciones clave:
- `tz_localize("zona")`: "este timestamp naive ya está en esta timezone" (no desplaza la hora)
- `tz_convert("zona")`: convierte de una timezone a otra (desplaza la hora)

In [ ]:
# Timestamp naive — sin información de zona horaria
ts_naive = pd.Timestamp("2024-06-01 10:00:00")
print(f"Naive timestamp: {ts_naive}")
print(f"tzinfo:          {ts_naive.tzinfo}  ← None = ambiguo!")
print()

# tz_localize: "este timestamp YA ES de esta timezone"
ts_cdmx = ts_naive.tz_localize("America/Mexico_City")
print(f"Localizado a CDMX: {ts_cdmx}")

# tz_convert: convertir a otra timezone
ts_utc = ts_cdmx.tz_convert("UTC")
ts_ny  = ts_cdmx.tz_convert("America/New_York")
ts_mad = ts_cdmx.tz_convert("Europe/Madrid")

print(f"→ UTC:            {ts_utc}")
print(f"→ Nueva York:     {ts_ny}")
print(f"→ Madrid:         {ts_mad}")
print()
print("Todos representan el MISMO momento en el tiempo.")

In [ ]:
# Lo mismo para una Serie completa
timestamps_locales = pd.Series(pd.date_range("2024-01-01", periods=6, freq="MS"))
print("Serie naive:")
print(timestamps_locales)
print()

# Localizar toda la Serie a CDMX y convertir a UTC
serie_cdmx = timestamps_locales.dt.tz_localize("America/Mexico_City")
serie_utc  = serie_cdmx.dt.tz_convert("UTC")

comparacion = pd.DataFrame({
    "naive": timestamps_locales,
    "CDMX (localizado)": serie_cdmx,
    "UTC (convertido)": serie_utc,
})
print("Comparación:")
print(comparacion.to_string())

### El problema del horario de verano (DST)

En los países que cambian de hora, hay momentos que **ocurren dos veces** (cuando el reloj se atrasa) o que **no existen** (cuando el reloj se adelanta). `tz_localize` lanza una excepción en esos casos.

En México (CDMX), el cambio de hora de noviembre 2024:
- El reloj retrocedió de 2:00 AM a 1:00 AM
- Por eso, las 1:30 AM del 3 de noviembre existieron **dos veces**

In [ ]:
# Este momento existió dos veces en CDMX el 3 de noviembre de 2024
ts_ambiguo = pd.Timestamp("2024-11-03 01:30:00")

# Intentar localizar sin indicar cuál de las dos ocurrencias
try:
    ts_ambiguo.tz_localize("America/Mexico_City")
except Exception as e:
    print(f"Error esperado: {type(e).__name__}")
    print(f"  {e}")

print()

# Solución: usar ambiguous="NaT" para marcar ambiguos como NaT
ts_nat = ts_ambiguo.tz_localize("America/Mexico_City", ambiguous="NaT")
print(f"Con ambiguous='NaT': {ts_nat}")

# O especificar si es la primera (DST=True) o segunda (DST=False) ocurrencia
ts_primera  = ts_ambiguo.tz_localize("America/Mexico_City", ambiguous=True)   # horario de verano
ts_segunda  = ts_ambiguo.tz_localize("America/Mexico_City", ambiguous=False)  # horario estándar
print(f"Primera ocurrencia:  {ts_primera}  → UTC: {ts_primera.tz_convert('UTC')}")
print(f"Segunda ocurrencia:  {ts_segunda}  → UTC: {ts_segunda.tz_convert('UTC')}")

In [ ]:
# En una Serie con múltiples timestamps alrededor del cambio de hora
# ambiguous="NaT" convierte los ambiguos en NaT en vez de lanzar error
timestamps_noviembre = pd.Series([
    "2024-11-03 00:00:00",
    "2024-11-03 01:00:00",
    "2024-11-03 01:30:00",  # ambiguo — existe dos veces
    "2024-11-03 02:00:00",  # ya en horario estándar
    "2024-11-03 03:00:00",
])
timestamps_noviembre = pd.to_datetime(timestamps_noviembre)

resultado_serie = timestamps_noviembre.dt.tz_localize(
    "America/Mexico_City", ambiguous="NaT"
)
print("Timestamps alrededor del cambio de hora (CDMX, noviembre 2024):")
print(resultado_serie)

> **💡 Prueba esto:** ¿Por qué es un problema si combinas eventos de CDMX y de Nueva York sin normalizar a UTC? Imagina que tienes logs de servidores: un evento en CDMX a las "2024-11-03 01:30:00" y uno en NYC a las "2024-11-03 01:30:00" — ¿son simultáneos? ¿Cuál ocurrió primero?

In [ ]:
#Si, es un problema, ya que se vuelve ambiguo porque puede referirse a instantes distintos según la ciudad, es como comparar una moneda con otra sin tomar en cuenta el tipo de cambio y únicamente el valor de la moneda en su respectiva ciudad
# Pueden ser simultaneos, pero tenemos que manejar las fechas a un mismo uso horario (ya sea UTC-6)
# El evento de NYC occurrió antes

cdmx = pd.Timestamp("2024-11-03 01:30:00").tz_localize("America/Mexico")
nyc = pd.Timestamp("2024-11-03 01:30:00").tz_localize("America/NewYork")

cdmx_utc = cdmx.tz_convert("UTC")
nyc_utc = cdmx.tz_convert("UTC")

print("¿NYC ocurrió antes que CDMX?",nyc_utc<cdmx_utc)

---
## Sección 5: Aritmética de fechas con `Timedelta` y `DateOffset`

Dos tipos para hacer aritmética con fechas:

- **`pd.Timedelta`**: duración fija en unidades absolutas (días, horas, segundos). No sabe nada de calendarios.
- **`pd.DateOffset`**: sigue el calendario. Sumar un mes siempre da el mismo día del mes siguiente, respetando el fin de mes.

La diferencia entre dos Timestamps siempre da un `Timedelta`.

In [ ]:
# Diferencia entre fechas → Timedelta
inicio = pd.Timestamp("2024-01-01")
fin    = pd.Timestamp("2024-12-31")

delta = fin - inicio
print(f"Delta: {delta}")
print(f"Días:     {delta.days}")
print(f"Segundos: {delta.seconds}")   # solo la parte de horas/min/seg, no días
print(f"Total en segundos: {delta.total_seconds():,.0f}")

In [ ]:
# Sumar Timedeltas
fecha_base = pd.Timestamp("2024-01-15 09:00:00")

print(f"Base:                {fecha_base}")
print(f"+ 7 días:            {fecha_base + pd.Timedelta(days=7)}")
print(f"+ 3h 30min:          {fecha_base + pd.Timedelta(hours=3, minutes=30)}")
print(f"+ 2 semanas:         {fecha_base + pd.Timedelta(weeks=2)}")
print()

# DateOffset — sigue el calendario
print(f"+ 1 mes (DateOffset): {fecha_base + pd.DateOffset(months=1)}")
print(f"+ 1 año (DateOffset): {fecha_base + pd.DateOffset(years=1)}")

# La diferencia importa en fin de mes
fin_enero = pd.Timestamp("2024-01-31")
print()
print(f"31-ene + 1 mes (DateOffset): {fin_enero + pd.DateOffset(months=1)}")  # 29-feb (2024 bisiesto)
print(f"31-ene + 31 días (Timedelta): {fin_enero + pd.Timedelta(days=31)}")  # 3-mar

In [ ]:
# Operaciones en columnas de un DataFrame
csv_str = """cliente,fecha_registro,fecha_vencimiento
A,2024-01-10,2024-04-10
B,2024-02-15,2024-05-15
C,2024-03-01,2024-06-01
D,2023-11-20,2024-02-20
E,2024-06-30,2024-09-30"""

df_clientes = pd.read_csv(StringIO(csv_str), parse_dates=["fecha_registro", "fecha_vencimiento"])

# Calcular métricas temporales
hoy = pd.Timestamp.now().normalize()  # normalize() quita la parte de hora

df_clientes["dias_como_cliente"] = (hoy - df_clientes["fecha_registro"]).dt.days
df_clientes["dias_hasta_vencimiento"] = (df_clientes["fecha_vencimiento"] - hoy).dt.days
df_clientes["duracion_contrato_dias"] = (
    df_clientes["fecha_vencimiento"] - df_clientes["fecha_registro"]
).dt.days
df_clientes["contrato_vencido"] = df_clientes["fecha_vencimiento"] < hoy

print(f"Fecha de referencia (hoy): {hoy.date()}\n")
print(df_clientes.to_string(index=False))

> **💡 Prueba esto:** Calcula la columna `"renovar_en_30_dias"` que sea `True` si el contrato vence dentro de los próximos 30 días (y aún no ha vencido). ¿Qué clientes necesitan renovar pronto?

In [ ]:
hoy = pd.Timestamp.now().normalize()

df_clientes["renovar_en_30_dias"] = (
    df_clientes["fecha_vencimiento"].ge(hoy) &
    df_clientes["fecha_vencimiento"].le(hoy + pd.Timedelta(days=30))
)

---
## Sección 6: `parse_dates` en `read_csv`

pandas tiene un parámetro `parse_dates` en `read_csv` que convierte columnas automáticamente. Sin embargo, tiene limitaciones importantes con formatos no estándar — y puede silenciosamente estar leyendo las fechas mal.

In [ ]:
# CSV con diferentes formatos de fecha
csv_iso = """evento,fecha
lanzamiento,2024-03-15
reunion,2024-04-01
entrega,2024-05-20"""

csv_latam = """evento,fecha
lanzamiento,15/03/2024
reunion,01/04/2024
entrega,20/05/2024"""

# ISO 8601: parse_dates funciona bien
df_iso = pd.read_csv(StringIO(csv_iso), parse_dates=["fecha"])
print("ISO 8601 con parse_dates:")
print(df_iso.dtypes)
print(df_iso)
print()

In [ ]:
# Formato latam: parse_dates puede fallar o malinterpretar
df_latam_auto = pd.read_csv(StringIO(csv_latam), parse_dates=["fecha"])
print("Formato latam (DD/MM/YYYY) con parse_dates automático:")
print(df_latam_auto)
print(f"dtype: {df_latam_auto['fecha'].dtype}")
print()

# Mejor práctica: leer como string, convertir explícitamente
df_latam_correcto = pd.read_csv(StringIO(csv_latam))
df_latam_correcto["fecha"] = pd.to_datetime(
    df_latam_correcto["fecha"], format="%d/%m/%Y"
)
print("Con to_datetime(format='%d/%m/%Y') explícito:")
print(df_latam_correcto)
print(f"dtype: {df_latam_correcto['fecha'].dtype}")

**Resumen de mejores prácticas:**

| Situación | Recomendación |
|-----------|---------------|
| Formato ISO 8601 (`YYYY-MM-DD`) | `parse_dates=["col"]` funciona bien |
| Formato ambiguo o no estándar | Leer como `str`, luego `pd.to_datetime(col, format="...")` |
| Unix timestamps | `pd.to_datetime(col.astype(int), unit="s")` |
| Datos de múltiples fuentes | Normalizar todo a UTC tan pronto como sea posible |

---
## Sección 7: Features temporales para ML

En un pipeline de ML necesitas convertir timestamps a features numéricas que el modelo pueda usar. La función siguiente extrae las más comunes de una sola vez.

In [ ]:
def extraer_features_tiempo(df, col_fecha):
    """Extrae features temporales útiles de una columna datetime.

    Devuelve una copia del DataFrame con columnas adicionales.
    """
    df = df.copy()
    dt = df[col_fecha].dt
    return df.assign(
        año=dt.year,
        mes=dt.month,
        dia=dt.day,
        dia_semana=dt.dayofweek,           # 0=lun, 6=dom
        es_fin_semana=(dt.dayofweek >= 5).astype(int),
        trimestre=dt.quarter,
        semana_año=dt.isocalendar().week.astype(int),
        hora=dt.hour,
        minuto=dt.minute,
        es_inicio_mes=(dt.day == 1).astype(int),
        es_fin_mes=(dt.day == dt.days_in_month).astype(int),
    )


# Probar con datos de ejemplo
np.random.seed(7)
df_ventas = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01 08:00", periods=10, freq="23h 47min"),
    "monto": np.random.randint(500, 10000, 10),
})

df_features = extraer_features_tiempo(df_ventas, "timestamp")
print("DataFrame con features temporales:")
print(df_features.to_string())

### Encoding cíclico: seno y coseno

El mes 12 y el mes 1 están a un paso de distancia, pero numéricamente la diferencia es 11. Para modelos lineales (y redes neuronales), esto crea un sesgo. La solución es convertir variables cíclicas a sus componentes seno y coseno:

$$\text{mes\_sin} = \sin\left(\frac{2\pi \cdot \text{mes}}{12}\right) \qquad \text{mes\_cos} = \cos\left(\frac{2\pi \cdot \text{mes}}{12}\right)$$

Con esto, mes 12 y mes 1 quedan cercanos en el espacio de features.

In [ ]:
# Encoding cíclico para mes y hora
df_features["mes_sin"] = np.sin(2 * np.pi * df_features["mes"] / 12)
df_features["mes_cos"] = np.cos(2 * np.pi * df_features["mes"] / 12)
df_features["hora_sin"] = np.sin(2 * np.pi * df_features["hora"] / 24)
df_features["hora_cos"] = np.cos(2 * np.pi * df_features["hora"] / 24)

# Mostrar que los meses extremos ahora están cerca
print("Distancia entre meses con encoding cíclico:")
mes_12_sin = np.sin(2 * np.pi * 12 / 12)
mes_12_cos = np.cos(2 * np.pi * 12 / 12)
mes_01_sin = np.sin(2 * np.pi * 1 / 12)
mes_01_cos = np.cos(2 * np.pi * 1 / 12)
mes_06_sin = np.sin(2 * np.pi * 6 / 12)
mes_06_cos = np.cos(2 * np.pi * 6 / 12)

dist_12_01 = np.sqrt((mes_12_sin - mes_01_sin)**2 + (mes_12_cos - mes_01_cos)**2)
dist_12_06 = np.sqrt((mes_12_sin - mes_06_sin)**2 + (mes_12_cos - mes_06_cos)**2)

print(f"  Distancia entre dic (12) y ene (1):  {dist_12_01:.4f}  ← cerca")
print(f"  Distancia entre dic (12) y jun (6):  {dist_12_06:.4f}  ← lejos")
print()
print("Columnas de features cíclicas:")
print(df_features[["timestamp", "mes", "mes_sin", "mes_cos", "hora", "hora_sin", "hora_cos"]].to_string())

> **💡 Prueba esto:** Agrega encoding cíclico para `dia_semana` (rango 0-6) a la función `extraer_features_tiempo`. ¿Cuánto vale la distancia entre lunes (0) y domingo (6) con encoding cíclico?

In [ ]:
#Encoding ciclico por día_semana (0.6)
df_features["dow_sin"] = np.sin(2 * np.pi * df_features["dia_semana"] / 7)
df_features["dow_cos"] = np.cos(2 * np.pi * df_features["dia_semana"] / 7)

lun_sin = np.sin(2 * np.pi * 0 / 7)
lun_cos = np.cos(2 * np.pi * 0 / 7)
dom_sin = np.sin(2 * np.pi * 6 / 7)
dom_cos = np.cos(2 * np.pi * 6 / 7)

dist_lun_dom = np.sqrt((lun_sin - dom_sin)**2 + (lun_cos - dom_cos)**2)

print(f"  Distancia entre lunes (0) y domingo (6): {dist_lun_dom:.4f}  ← cerca")


---
## Sección 8: Ejercicio

Tienes un DataFrame con eventos que ocurrieron en distintas ciudades, con sus timestamps en hora local. Tu tarea:

1. Convertir todos los timestamps a UTC
2. Extraer features temporales: hora, día de semana, mes
3. Identificar los eventos que ocurrieron en fin de semana (UTC)
4. Calcular cuántos días hace que ocurrió cada evento

**Datos de entrada:**

In [ ]:
# Datos del ejercicio — NO modificar esta celda
datos_eventos = pd.DataFrame({
    "evento": ["login", "compra", "logout", "registro", "consulta", "pago"],
    "timestamp_local": [
        "2024-03-10 09:15:00",
        "2024-03-15 14:30:00",
        "2024-03-16 23:45:00",
        "2024-03-17 06:00:00",
        "2024-03-22 11:20:00",
        "2024-03-23 18:55:00",
    ],
    "ciudad": ["CDMX", "Nueva York", "CDMX", "Madrid", "Nueva York", "CDMX"],
    "timezone": [
        "America/Mexico_City",
        "America/New_York",
        "America/Mexico_City",
        "Europe/Madrid",
        "America/New_York",
        "America/Mexico_City",
    ],
})

datos_eventos["timestamp_local"] = pd.to_datetime(datos_eventos["timestamp_local"])
print("Datos de entrada:")
print(datos_eventos.to_string(index=False))

In [ ]:
df_sol = datos_eventos.copy()

df_sol["timestamp_utc"] = df_sol.apply(
    lambda row: row["timestamp_local"].tz_localize(row["timezone"]).tz_convert("UTC"),
    axis=1,
)

df_sol["hora_utc"] = df_sol["timestamp_utc"].dt.hour
df_sol["dia_semana_utc"] = df_sol["timestamp_utc"].dt.dayofweek
df_sol["mes_utc"] = df_sol["timestamp_utc"].dt.month

df_sol["es_fin_semana"] = df_sol["timestamp_utc"].dt.dayofweek >= 5

ahora_utc = pd.Timestamp.now(tz="UTC")
df_sol["dias_desde_evento"] = (ahora_utc - df_sol["timestamp_utc"]).dt.days

print(df_sol.to_string(index=False))


<details>
<summary><strong>Ver solución</strong></summary>

```python
df_sol = datos_eventos.copy()

# Paso 1: Convertir cada timestamp local a UTC
def localizar_y_convertir_utc(row):
    ts = row["timestamp_local"]
    tz = row["timezone"]
    return ts.tz_localize(tz).tz_convert("UTC")

df_sol["timestamp_utc"] = df_sol.apply(localizar_y_convertir_utc, axis=1)

# Paso 2: Extraer features temporales en UTC
df_sol["hora_utc"]      = df_sol["timestamp_utc"].dt.hour
df_sol["dia_semana_utc"] = df_sol["timestamp_utc"].dt.dayofweek
df_sol["mes_utc"]       = df_sol["timestamp_utc"].dt.month

# Paso 3: Fin de semana en UTC
df_sol["es_fin_semana"] = df_sol["timestamp_utc"].dt.dayofweek >= 5

# Paso 4: Días desde el evento
# pd.Timestamp.now(tz="UTC") para que tenga la misma timezone que timestamp_utc
ahora_utc = pd.Timestamp.now(tz="UTC")
df_sol["dias_desde_evento"] = (ahora_utc - df_sol["timestamp_utc"]).dt.days

print("Resultado:")
print(df_sol[["evento", "ciudad", "timestamp_utc", "hora_utc",
              "dia_semana_utc", "es_fin_semana", "dias_desde_evento"]].to_string(index=False))

print(f"\nEventos en fin de semana (UTC):")
print(df_sol.loc[df_sol["es_fin_semana"], ["evento", "ciudad", "timestamp_utc"]].to_string(index=False))
```
</details>

---
## Resumen

| Operación | Código |
|-----------|--------|
| Parsear fechas con formato ambiguo | `pd.to_datetime(s, format="%d/%m/%Y")` |
| Ignorar valores inválidos | `pd.to_datetime(s, errors="coerce")` → `NaT` |
| Timestamp actual | `pd.Timestamp.now()` |
| Secuencia de fechas | `pd.date_range("2024-01-01", periods=12, freq="MS")` |
| Extraer componentes | `s.dt.year`, `.dt.month`, `.dt.dayofweek`, `.dt.hour`, ... |
| Localizar timezone | `ts.tz_localize("America/Mexico_City")` |
| Convertir timezone | `ts.tz_convert("UTC")` |
| Diferencia entre fechas | `ts2 - ts1` → `Timedelta` |
| Sumar duración fija | `fecha + pd.Timedelta(days=7)` |
| Sumar por calendario | `fecha + pd.DateOffset(months=1)` |
| Features ML | `df.assign(es_fin_semana=(dt.dayofweek >= 5).astype(int), ...)` |
| Encoding cíclico | `np.sin(2 * np.pi * mes / 12)` |